# Chapter 1 ASIC XGBoost Recalibration Review

This notebook reruns post hoc recalibration from saved ASIC baseline prediction artifacts only.
It prefers exported cluster prediction artifacts when available locally and falls back to synthetic local baseline artifacts otherwise.
It fits recalibration on the validation split, applies the fitted mappings to test, and compares raw XGBoost, recalibrated XGBoost, and logistic regression.

Embedded outputs, if present, reflect the last time the notebook was executed and may predate the current artifact-source resolution rules.


## Setup

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

try:
    from IPython.display import Image, Markdown, display
except ImportError:
    class Markdown(str):
        pass

    class Image:
        def __init__(self, filename: str):
            self.filename = filename

        def __repr__(self) -> str:
            return f"Image(filename={self.filename!r})"

    def display(*objects):
        for obj in objects:
            print(obj)


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not locate the repository root from the current working directory.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from chapter1_mortality_decomposition.notebook_artifact_sources import (
    build_artifact_source_markdown,
    notebook_artifact_source,
)
from chapter1_mortality_decomposition.xgboost_recalibration import (
    DEFAULT_RECALIBRATION_OUTPUT_DIR,
    run_asic_xgboost_recalibration,
)


def display_path(path: Path) -> str:
    try:
        return str(Path(path).resolve().relative_to(PROJECT_ROOT.resolve()))
    except ValueError:
        return str(Path(path).resolve())


PROJECT_ROOT


## Artifact Loading And Recalibration Run

In [ ]:
output_dir = PROJECT_ROOT / DEFAULT_RECALIBRATION_OUTPUT_DIR

result = run_asic_xgboost_recalibration(
    output_dir=output_dir,
)

input_source = notebook_artifact_source(
    "Baseline prediction exports",
    path=result.input_root,
    resolution=result.input_root_resolution,
)
display(Markdown(build_artifact_source_markdown([input_source], display_root=PROJECT_ROOT)))
display(Markdown(f"**Output directory:** `{display_path(result.output_dir)}`"))
display(Markdown(f"**Horizons processed:** `{', '.join(str(h) for h in result.horizons_processed)}`"))

status_rows = []
for horizon_result in result.horizon_results:
    row = {"horizon_h": horizon_result.horizon_h}
    for method_name, status in sorted(horizon_result.method_statuses.items()):
        row[f"{method_name}_status"] = status
    status_rows.append(row)
pd.DataFrame(status_rows).sort_values("horizon_h").reset_index(drop=True)


## Metric Comparison

In [ ]:
metrics = pd.read_csv(result.combined_comparison_metrics_path)
metric_columns = [
    "horizon_h",
    "model_variant",
    "split",
    "sample_count",
    "event_count",
    "event_rate",
    "auroc",
    "auprc",
    "calibration_intercept",
    "calibration_slope",
    "brier_score",
    "mean_predicted_risk",
    "notes",
]

validation_metrics = (
    metrics.loc[metrics["split"].eq("validation"), metric_columns]
    .sort_values(["horizon_h", "model_variant"])
    .reset_index(drop=True)
)
test_metrics = (
    metrics.loc[metrics["split"].eq("test"), metric_columns]
    .sort_values(["horizon_h", "model_variant"])
    .reset_index(drop=True)
)

display(Markdown("### Validation Metrics"))
display(validation_metrics)
display(Markdown("### Test Metrics"))
display(test_metrics)

In [ ]:
test_brier_summary = (
    test_metrics[
        ["horizon_h", "model_variant", "event_rate", "mean_predicted_risk", "brier_score"]
    ]
    .pivot(index="horizon_h", columns="model_variant", values=["mean_predicted_risk", "brier_score"])
    .sort_index()
)
test_brier_summary

## Reliability Visualization

In [ ]:
display(Markdown("### Cross-horizon summary figure"))
display(Image(filename=str(result.summary_figure_path)))

for horizon_result in result.horizon_results:
    display(Markdown(f"### Horizon {horizon_result.horizon_h}h"))
    display(Image(filename=str(horizon_result.reliability_plot_path)))
    display(Image(filename=str(horizon_result.probability_distribution_plot_path)))

## Interpretation Note

In [ ]:
display(Markdown((result.interpretation_note_path).read_text()))